# E-Commerce Sales Analysis & Management Dashboard
**Data Analyst Project 01**  
**Dataset**: Olist Brazilian E-Commerce Dataset

## Project Overview
This project analyzes 12 months of sales data (from **September 2017 to August 2018**) from the Brazilian e-commerce platform Olist. The objectives are:
1. **Data Cleaning**: Load and merge multiple Olist datasets, filter out cancelled/unavailable orders, handle duplicates, translate category names to English, and document cleaning decisions.
2. **Sales Analysis**: Answer five key business performance questions (top categories, peak sales, regional performance, average order value, and review scores).
3. **Visualisation**: Generate 6 key charts showing revenue, trends, geography, and satisfaction.
4. **Dashboard**: Construct an executive-level summary dashboard in Excel.
5. **Insights Report**: Write a business report with 5 actionable recommendations.

### 1. Project Setup & Package Imports
Let's import the required Python packages for data manipulation, analysis, and plotting.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Configure plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 16
})

PALETTE_PRIMARY = "#1F4E78"
print("Pandas version:", pd.__version__)
print("Matplotlib version:", plt.matplotlib.__version__)
print("Seaborn version:", sns.__version__)

### 2. Data Loading & Merging
We load the CSV files, check shapes, and perform data merging. We will merge:
- `olist_orders_dataset.csv` (Main orders table)
- `olist_order_items_dataset.csv` (Product items per order)
- `olist_products_dataset.csv` (Product details)
- `product_category_name_translation.csv` (Portuguese to English translation)
- `olist_customers_dataset.csv` (Customer state/city)
- `olist_order_reviews_dataset.csv` (Review scores)

#### Data Cleaning Decisions:
1. **Order Filtering**: Exclude `canceled` and `unavailable` orders, as they do not represent realized revenue.
2. **Datetime Parsing**: Convert date strings to datetime objects.
3. **Timeframe Selection**: Filter orders to **September 2017 to August 2018 inclusive** (exactly 12 months) where sales data is most stable, complete, and representative.
4. **Deduplicate Reviews**: Calculate the *average* review score per `order_id` prior to joining, preventing artificial duplication of item records.
5. **Translate Categories**: Use English translations; fill missing/untranslated category names with 'Unknown' or clean formatting.
6. **Handle Missing Review Scores**: Fill missing scores with the overall average review score.

In [ ]:
# Define file paths
DATA_DIR = "."
files = {
    'orders': os.path.join(DATA_DIR, 'olist_orders_dataset.csv'),
    'items': os.path.join(DATA_DIR, 'olist_order_items_dataset.csv'),
    'products': os.path.join(DATA_DIR, 'olist_products_dataset.csv'),
    'reviews': os.path.join(DATA_DIR, 'olist_order_reviews_dataset.csv'),
    'customers': os.path.join(DATA_DIR, 'olist_customers_dataset.csv'),
    'translation': os.path.join(DATA_DIR, 'product_category_name_translation.csv')
}

# Load raw files
df_orders = pd.read_csv(files['orders'])
df_items = pd.read_csv(files['items'])
df_products = pd.read_csv(files['products'])
df_reviews = pd.read_csv(files['reviews'])
df_customers = pd.read_csv(files['customers'])
df_translation = pd.read_csv(files['translation'], encoding='utf-8-sig')

print(f"Raw Orders shape: {df_orders.shape}")
print(f"Raw Items shape: {df_items.shape}")

# Cleaning Decisions implementation
# 1. Exclude canceled and unavailable orders
active_orders = df_orders[~df_orders['order_status'].isin(['canceled', 'unavailable'])].copy()

# 2. Parse datetime columns
for col in ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']:
    active_orders[col] = pd.to_datetime(active_orders[col])

# 3. Filter to 12 months (2017-09 to 2018-08)
start_date = pd.to_datetime('2017-09-01 00:00:00')
end_date = pd.to_datetime('2018-08-31 23:59:59')
df_orders_12m = active_orders[
    (active_orders['order_purchase_timestamp'] >= start_date) & 
    (active_orders['order_purchase_timestamp'] <= end_date)
].copy()

# 4. Deduplicate review scores per order_id
df_reviews_unique = df_reviews.groupby('order_id', as_index=False).agg({'review_score': 'mean'})

# 5. Clean translation column names
df_translation.columns = df_translation.columns.str.replace(r'^\ufeff', '', regex=True)

# Merging datasets
merged = df_orders_12m.merge(df_items, on='order_id', how='inner')
merged = merged.merge(df_products, on='product_id', how='left')
merged = merged.merge(df_translation, on='product_category_name', how='left')

# Handle missing/untranslated category names
merged['product_category_name_english'] = merged['product_category_name_english'].fillna('Unknown')
missing_trans = (merged['product_category_name_english'] == 'Unknown') & (merged['product_category_name'].notna())
merged.loc[missing_trans, 'product_category_name_english'] = merged.loc[missing_trans, 'product_category_name'].apply(
    lambda x: str(x).replace('_', ' ').title()
)

merged = merged.merge(df_customers, on='customer_id', how='left')
merged = merged.merge(df_reviews_unique, on='order_id', how='left')
merged['review_score'] = merged['review_score'].fillna(merged['review_score'].mean())
merged['purchase_month'] = merged['order_purchase_timestamp'].dt.strftime('%Y-%m')

print(f"Final Cleaned & Merged dataset shape: {merged.shape}")

### 3. Sales Analysis (5 Questions Answered)
Let's write queries to answer the five manager questions.

In [ ]:
print("=== Q1: Which product category has the highest revenue? ===")
category_revenue = merged.groupby('product_category_name_english')['price'].sum().sort_values(ascending=False)
print(f"Highest Revenue Category: '{category_revenue.index[0]}' with R$ {category_revenue.iloc[0]:,.2f}\n")

print("=== Q2: Which month had peak sales? ===")
monthly_revenue = merged.groupby('purchase_month')['price'].sum().sort_values(ascending=False)
print(f"Peak Sales Month: {monthly_revenue.index[0]} with R$ {monthly_revenue.iloc[0]:,.2f}\n")

print("=== Q3: Which region performs best? ===")
state_revenue = merged.groupby('customer_state')['price'].sum().sort_values(ascending=False)
top_st = state_revenue.index[0]
top_st_sales = state_revenue.iloc[0]
top_st_pct = (top_st_sales / merged['price'].sum()) * 100
print(f"Top Performing Region (Customer State): {top_st} with R$ {top_st_sales:,.2f} ({top_st_pct:.1f}% of total)\n")

print("=== Q4: What is the average order value trend? ===")
monthly_aov = merged.groupby('purchase_month').agg(
    revenue=('price', 'sum'),
    orders=('order_id', 'nunique')
)
monthly_aov['aov'] = monthly_aov['revenue'] / monthly_aov['orders']
print(monthly_aov[['orders', 'revenue', 'aov']].sort_index(), "\n")

print("=== Q5: What is the customer review score distribution? ===")
df_order_reviews = merged.drop_duplicates('order_id')
review_dist = df_order_reviews['review_score'].round().value_counts().sort_index()
for score, count in review_dist.items():
    pct = (count / df_order_reviews.shape[0]) * 100
    print(f" - {int(score)} Stars: {count:,} orders ({pct:.1f}%)")

### 4. Visualisations
We display the 6 charts generated during our analysis.

In [ ]:
# Let's display the saved high-resolution visualisations inline in our notebook
v_files = [
    "visualizations/01_revenue_by_category.png",
    "visualizations/02_monthly_sales_trend.png",
    "visualizations/03_regional_sales.png",
    "visualizations/04_order_values_distribution.png",
    "visualizations/05_review_scores_distribution.png",
    "visualizations/06_category_monthly_heatmap.png"
]

for v_file in v_files:
    print(f"\n{'='*20} {os.path.basename(v_file)} {'='*20}")
    if os.path.exists(v_file):
        display(Image(filename=v_file))
    else:
        print(f"File not found: {v_file}")

### 5. Business Insights & Recommendations

Here are 5 core actionable insights and recommendations derived from this analysis:

1. **Capitalize on Health & Beauty Revenue Dominance**
   - *Finding*: Health & Beauty is the highest revenue-generating category (R$ 1.00M, 9.7% share), as seen in [01_revenue_by_category.png](file:///c:/Users/Lenovo/Desktop/PROJECT%2001%20E-Commerce%20Sales%20Analysis/visualizations/01_revenue_by_category.png).
   - *Recommendation*: Allocate more marketing budget towards Health & Beauty and introduce loyalty rewards or bundle deals in this category to increase customer lifetime value.

2. **Implement Holiday & Black Friday Campaigns Early**
   - *Finding*: November 2017 is the peak month by sales (R$ 1.00M revenue, 7.4k orders), as shown in the [02_monthly_sales_trend.png](file:///c:/Users/Lenovo/Desktop/PROJECT%2001%20E-Commerce%20Sales%20Analysis/visualizations/02_monthly_sales_trend.png) trend line.
   - *Recommendation*: Run early-bird campaigns leading up to November and ensure warehouse and carrier capacities are scaled to handle a 40%+ spike in order volume.

3. **Establish Regional Hubs in High-Performing States**
   - *Finding*: The State of São Paulo (SP) dominates with 39% of total revenue (R$ 4.04M), followed by RJ and MG, as seen in [03_regional_sales.png](file:///c:/Users/Lenovo/Desktop/PROJECT%2001%20E-Commerce%20Sales%20Analysis/visualizations/03_regional_sales.png).
   - *Recommendation*: Establish localized inventory hubs in São Paulo to offer same-day/next-day shipping, reducing freight costs and delivery times to maintain dominance.

4. **Boost AOV with Threshold-Based Free Shipping**
   - *Finding*: The median order value is R$ 90.00 while the overall average is R$ 137.62. Additionally, the AOV dropped by R$ 15.00 during the November peak, as shown in [04_order_values_distribution.png](file:///c:/Users/Lenovo/Desktop/PROJECT%2001%20E-Commerce%20Sales%20Analysis/visualizations/04_order_values_distribution.png) and the AOV table.
   - *Recommendation*: Introduce threshold-based free shipping (e.g., "Free Shipping on orders above R$ 120.00") to incentivize customers to add more items, raising the median order value.

5. **Address the Customer Satisfaction Gap (11.1% 1-Star Reviews)**
   - *Finding*: While 57.6% of orders received a 5-star rating, a concerning 11.1% of reviews are 1-star ratings, as shown in the [05_review_scores_distribution.png](file:///c:/Users/Lenovo/Desktop/PROJECT%2001%20E-Commerce%20Sales%20Analysis/visualizations/05_review_scores_distribution.png) donut chart.
   - *Recommendation*: Investigate the root causes of 1-star reviews (typically related to shipping delays or damaged packages). Implement automatic email follow-ups for low ratings and offer discounts to recover unhappy customers.

### Most Surprising Finding (Short Note)
The most surprising finding is that during the **November 2017 peak (Black Friday)**, despite total orders spiking by over **63%** (from 4,547 to 7,421) and total revenue reaching R$ 1.00M, the **Average Order Value (AOV) actually dropped to R$ 135.27** (down from R$ 145.19 in October). This indicates that the volume spike was driven by customers purchasing lower-priced discounted items rather than larger bulk orders, suggesting that promotions should focus on upselling complementary items during peak sales.